In [1]:
!nvidia-smi

Tue Jul 28 07:09:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
inputDataset = "/content/drive/MyDrive/PKLot.v2-640.yolov8"
outputDataset = "/content/drive/MyDrive/PKLot_Preprocessed"

In [4]:
import cv2
import numpy as np
import os
import shutil
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

In [5]:
# Create CLAHE object once
clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def preprocessImage(image):
    # Step 1: Noise Reduction
    image = cv2.GaussianBlur(image,(3,3),0)
    # Step 2: CLAHE Contrast Enhancement
    lab = cv2.cvtColor(image,cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    image = cv2.cvtColor(lab,cv2.COLOR_LAB2BGR)
    return image

In [6]:
def processSingleImage(args):
    imagePath, outputPath = args
    image = cv2.imread(imagePath)
    if image is None:
        return False
    image = preprocessImage(image)
    cv2.imwrite(outputPath,image)
    return True

In [7]:
splits = ["train","valid","test"]
# Available CPU cores
workers = cpu_count()
print("CPU cores available:", workers)

for split in splits:
    inputFolder = os.path.join(inputDataset,split,"images")
    outputFolder = os.path.join(outputDataset,split,"images")
    os.makedirs(outputFolder,exist_ok=True)
    imageNames = os.listdir(inputFolder)
    tasks = []
    for imageName in imageNames:
        inputPath = os.path.join(inputFolder,imageName)
        outputPath = os.path.join(outputFolder,imageName)
        tasks.append((inputPath,outputPath))
    print("\nProcessing:", split)
    with Pool(workers) as pool:
        results = list(tqdm(pool.imap(processSingleImage,tasks),total=len(tasks)))
    print(split, "images completed")

CPU cores available: 2

Processing: train


100%|██████████| 8702/8702 [07:43<00:00, 18.79it/s]


train images completed

Processing: valid


100%|██████████| 2484/2484 [02:42<00:00, 15.30it/s]


valid images completed

Processing: test


100%|██████████| 1242/1242 [01:29<00:00, 13.95it/s]

test images completed


In [8]:
for split in splits:
    sourceLabels = os.path.join(inputDataset,split,"labels")
    destinationLabels = os.path.join(outputDataset,split,"labels")
    shutil.copytree(sourceLabels,destinationLabels,dirs_exist_ok=True)

    print(split, "labels copied")

train labels copied
valid labels copied
test labels copied


In [11]:
for split in splits:

    imagePath = os.path.join(outputDataset,split,"images")
    labelPath = os.path.join(outputDataset,split,"labels")
    print("\n", split)
    print("Images:",len(os.listdir(imagePath)))
    print("Labels:",len(os.listdir(labelPath)))


 train
Images: 8702
Labels: 8691

 valid
Images: 2484
Labels: 2483

 test
Images: 1242
Labels: 1242


In [12]:
splits = ["train", "valid", "test"]
for split in splits:

    imageFolder = os.path.join(
        outputDataset,
        split,
        "images"
    )

    labelFolder = os.path.join(
        outputDataset,
        split,
        "labels"
    )
    images = os.listdir(imageFolder)
    missingLabels = []
    for image in images:

        imageName = os.path.splitext(image)[0]

        labelPath = os.path.join(
            labelFolder,
            imageName + ".txt"
        )
        if not os.path.exists(labelPath):
            missingLabels.append(image)

    print(split)
    print("Missing labels:", len(missingLabels))
    if len(missingLabels) > 0:
        print(missingLabels[:5])

train
Missing labels: 11
['2013-03-14_13_06_00_jpg.rf.4b3682c4ab4f3991b87d43bfdd344055 (1).jpg', '2013-03-14_12_50_59_jpg.rf.eb598b1be8a427a940277d6fb21f5019 (1).jpg', '2013-03-14_12_20_59_jpg.rf.6f8d4e4f9708449a8f85fac03381b429 (1).jpg', '2013-03-14_13_16_00_jpg.rf.68dd1835c809e805164a4ebc6fb8d598 (1).jpg', '2013-03-14_12_30_59_jpg.rf.f8ed5ecdf317e4bb7ea4b692ceeea753 (1).jpg']
valid
Missing labels: 1
['2012-10-13_17_19_02_jpg.rf.e49f0792ca2ea9b394c180007a11e4d7 (1).jpg']
test
Missing labels: 0


In [13]:
for split in splits:
    imageFolder = os.path.join(
        outputDataset,
        split,
        "images"
    )
    labelFolder = os.path.join(
        outputDataset,
        split,
        "labels"
    )

    for image in os.listdir(imageFolder):
        imageName = os.path.splitext(image)[0]
        labelPath = os.path.join(
            labelFolder,
            imageName + ".txt"
        )
        if not os.path.exists(labelPath):
            imagePath = os.path.join(
                imageFolder,
                image
            )
            os.remove(imagePath)
            print("Removed:", image)

Removed: 2013-03-14_13_06_00_jpg.rf.4b3682c4ab4f3991b87d43bfdd344055 (1).jpg
Removed: 2013-03-14_12_50_59_jpg.rf.eb598b1be8a427a940277d6fb21f5019 (1).jpg
Removed: 2013-03-14_12_20_59_jpg.rf.6f8d4e4f9708449a8f85fac03381b429 (1).jpg
Removed: 2013-03-14_13_16_00_jpg.rf.68dd1835c809e805164a4ebc6fb8d598 (1).jpg
Removed: 2013-03-14_12_30_59_jpg.rf.f8ed5ecdf317e4bb7ea4b692ceeea753 (1).jpg
Removed: 2013-03-14_12_25_59_jpg.rf.651aff8a612bb8b3cd9069a03afebd1a (1).jpg
Removed: 2012-09-12_10_32_11_jpg.rf.16afb0c3d046037ff4b02b0a53cb9306 (1).jpg
Removed: 2012-09-16_09_58_04_jpg.rf.f2ae82e4ad93d93c62f3bf5889097cdb (1).jpg
Removed: 2012-09-16_10_28_05_jpg.rf.01fcbe8edd92363275488128f5fa4d4d (1).jpg
Removed: 2012-09-16_10_13_04_jpg.rf.77d91694c4d9ab5b92ba62772c0e913f (1).jpg
Removed: 2012-09-16_10_18_05_jpg.rf.58d90cc0c8ffc566cecb1cf32e83f542 (1).jpg
Removed: 2012-10-13_17_19_02_jpg.rf.e49f0792ca2ea9b394c180007a11e4d7 (1).jpg


In [17]:
sourceYaml = os.path.join(inputDataset,"data.yaml")
destinationYaml = os.path.join(outputDataset,"data.yaml")
shutil.copy(sourceYaml,destinationYaml)
print("data.yaml copied")

data.yaml copied


In [18]:
yamlPath = os.path.join(outputDataset,"data.yaml")
print(open(yamlPath).read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['space-empty', 'space-occupied']

roboflow:
  workspace: brad-dwyer
  project: pklot-1tros
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/brad-dwyer/pklot-1tros/dataset/2


In [19]:
yamlPath = "/content/drive/MyDrive/PKLot_Preprocessed/data.yaml"

yamlContent = """
train: /content/drive/MyDrive/PKLot_Preprocessed/train/images
val: /content/drive/MyDrive/PKLot_Preprocessed/valid/images
test: /content/drive/MyDrive/PKLot_Preprocessed/test/images

nc: 2
names: ['space-empty', 'space-occupied']

roboflow:
  workspace: brad-dwyer
  project: pklot-1tros
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/brad-dwyer/pklot-1tros/dataset/2
"""

with open(yamlPath, "w") as file:
    file.write(yamlContent)

print("data.yaml updated successfully!")

data.yaml updated successfully!


In [20]:
with open(yamlPath, "r") as file:
    print(file.read())


train: /content/drive/MyDrive/PKLot_Preprocessed/train/images
val: /content/drive/MyDrive/PKLot_Preprocessed/valid/images
test: /content/drive/MyDrive/PKLot_Preprocessed/test/images

nc: 2
names: ['space-empty', 'space-occupied']

roboflow:
  workspace: brad-dwyer
  project: pklot-1tros
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/brad-dwyer/pklot-1tros/dataset/2



In [21]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(
    data="/content/drive/MyDrive/PKLot_Preprocessed/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/content/drive/MyDrive/YOLO_Training",
    name="PKLot_Model"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/PKLot_Preprocessed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fl